In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_testing_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_0105_064335', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_0105_075306', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_0105_090239', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_0105_101212', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_0105_112151', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_123134', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_0105_134115', 'chirps_precipitation_mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_0105_145050']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:         (date: 2191, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 9kB 0.34 0.31 0.27 ... nan nan nan
       streamflow_sim  (date, time_step) float32 9kB 0.3677 0.3646 ... 1.088 1.024}},
 'camels_01466500': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:         (date: 2191, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 9kB 0.34 0.33 0.33 ... nan nan nan
       streamflow_sim  (date, time_step) float32 9kB 0.4908 0.4813 ... 0.6977}},
 'camels_01487000': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:         (date: 2191, time_step: 1)
   Coordinates:

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}

for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['streamflow_obs']
    sim = xr_ds['streamflow_sim']
    
    # Skip basin if all obs or sim are NaN
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = calculate_metrics(
        obs=obs,
        sim=sim,
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.731182,0.370501,0.608688,0.865683,1.035878,0.870922,1.009620,0.010533,1.781391,-1.164443,25.965523,0.333333,0.103448,28.803787
camels_01466500,0.537620,0.156700,0.395853,0.766156,1.111302,0.799852,1.047283,0.071706,12.783276,-10.552432,35.106220,0.363636,0.520000,27.183243
camels_01487000,0.641917,0.585415,0.765124,0.808157,0.963700,0.815679,1.038882,0.038806,-9.387368,-1.186690,-51.906361,0.230769,0.360000,40.167320
camels_01638480,0.705109,2.276089,1.508671,0.690003,0.765546,0.845299,0.868864,-0.055324,-28.512445,-21.741333,-414.824829,0.230769,0.281250,43.618359
camels_01644000,0.677245,1.179417,1.086010,0.803930,0.918490,0.828241,1.047942,0.024360,-7.230554,-14.762674,10.948265,0.250000,0.322581,32.120819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_09505800,0.270644,0.081121,0.284818,0.581619,0.906817,0.603508,0.904328,-0.039731,-18.839233,106.332863,-643.986389,1.000000,0.666667,72.538651
camels_10259000,-1.205026,0.305271,0.552514,-0.260855,2.247245,0.856379,1.116223,0.062359,59.103401,0.293999,79.340698,0.666667,0.541667,108.860825
camels_11451100,0.823012,1.631229,1.277196,0.902741,0.993883,0.911080,1.038925,0.014114,-4.026783,-17.904602,74.459343,0.571429,0.285714,40.960358


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(ensemble_metrics_dir/f"{save_name}.csv")

In [7]:
df_metrics.median()

NSE              0.732674
MSE              0.812095
RMSE             0.901163
KGE              0.748753
Alpha-NSE        0.919732
Pearson-r        0.882683
Beta-KGE         1.073093
Beta-NSE         0.033332
FHV             -7.230554
FMS            -21.741333
FLV             10.383559
Peak-Timing      0.333333
Missed-Peaks     0.344828
Peak-MAPE       40.322105
dtype: float64